In [1]:
from pathlib import Path
import os, sys
if os.environ.get("GDAL_DATA") is None:
    env_root = Path(sys.executable).parents[1]
    os.environ["GDAL_DATA"] = str(env_root / "Library" / "share" / "gdal")
if "PROJ_LIB" not in os.environ:
    env_root = Path(sys.executable).parents[1]
    os.environ["PROJ_LIB"] = str(env_root / "Library" / "share" / "proj")
import fiona
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import shape, box


In [2]:
# paths

project_root = Path().resolve().parent.parent

data_folder = project_root / "data" / "csen12" / "preprocess_test"
aoi_file = project_root / "data" / "aoi_boundary" / "selected_countries.gpkg"
list_file = project_root / "data" / "csen12" / "preprocess_test" /  "rasters_h_in_aoi_test.txt"

In [3]:
# load separate AOI polygons 
# (maybe not only deserted areas in the future - to test influence on model performance) 

with fiona.open(aoi_file) as aoi:
    aoi_crs = aoi.crs_wkt
    aoi_polygons = [shape(polygon["geometry"]) for polygon in aoi]

In [4]:
# iterate over rasters

selected_rasters = []

# cache AOI bounding boxes per raster CRS
aoi_crs_check = {}

for raster_path in data_folder.rglob("*.tif"):
    with rasterio.open(raster_path) as raster:
        raster_bounds = raster.bounds
        raster_crs = raster.crs

    raster_box = box(*raster_bounds)

    # build AOI box in each raster crs
    if raster_crs not in aoi_crs_check:
        aoi_box = []
        for aoi_polygon in aoi_polygons:
            bounds_in_raster_crs = transform_bounds(
                aoi_crs,
                raster_crs,
                *aoi_polygon.bounds
            )
            # create bounding box from tuple bound coordinates
            aoi_box.append(box(*bounds_in_raster_crs))

        aoi_crs_check[raster_crs] = aoi_box

    # test intersection with aoi
    for aoi_bbox in aoi_crs_check[raster_crs]:
        if raster_box.intersects(aoi_bbox):
            selected_rasters.append(raster_path)
            break

print(f"selected rasters: {len(selected_rasters)}")


with list_file.open("w", encoding="utf-8") as f:
    for path in selected_rasters:
        f.write(f"{path}\n")

selected rasters: 10


In [3]:
import fiona
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import shape, box
from pathlib import Path


def select_intersecting_rasters(data_folder, aoi_file, list_file):
    with fiona.open(aoi_file) as aoi:
        aoi_crs = aoi.crs_wkt
        aoi_polygons = [shape(polygon["geometry"]) for polygon in aoi]

    selected = []
    aoi_crs_check = {}

    for raster_path in data_folder.rglob("*.tif"):
        with rasterio.open(raster_path) as raster:
            raster_bounds = raster.bounds
            raster_crs = raster.crs

        raster_box = box(*raster_bounds)

        if raster_crs not in aoi_crs_check:
            aoi_boxes = []
            for aoi_polygon in aoi_polygons:
                bounds_in_raster_crs = transform_bounds(aoi_crs, raster_crs, *aoi_polygon.bounds)
                aoi_boxes.append(box(*bounds_in_raster_crs))
            aoi_crs_check[raster_crs] = aoi_boxes

        if any(raster_box.intersects(aoi_bbox) for aoi_bbox in aoi_crs_check[raster_crs]):
            selected.append(raster_path)

    list_file.write_text("\n".join(str(p) for p in selected), encoding="utf-8")
    print(f"selected rasters: {len(selected)}")
    return selected

In [5]:
select_intersecting_rasters(data_folder, aoi_file, list_file)

selected rasters: 270


[WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0012/20180116T091319_20180116T091445_T34RFV.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0012/20180302T090901_20180302T092002_T34RFV.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0012/20180829T090551_20180829T091949_T34RFV.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0012/20181112T091219_20181112T092057_T34RFV.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0012/20191217T091309_20191217T092124_T34RFV.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0016/20180218T083011_20180218T083855_T36RUT.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0016/20181115T083211_20181115T083248_T36RUT.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0016/20181125T083251_20181125T083250_T36RUT.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0016/20191120T083241_20191120T083254_T36RUT.tif'),
 WindowsPath('C:/skola/diplomka/data/csen12/high/ROI_0016/202003